ResNet-18 中的数字 18，指的是通常统计的带权重层数：17 个卷积层加上 1 个全连接层。它不包含池化层、BatchNorm 和 ReLU 等操作。四组残差层各有两个 BasicBlock，每个 BasicBlock 包含两个卷积层。因此残差部分共有 4*2*2=16 个卷积层，再加上最开始的 1 个卷积层和最后的 1 个全连接层，总计 18 层。

ResNet-18 原始设计常以 224×224 图像为输入，但 CIFAR-10 图片只有 32×32。

如果直接照搬原始网络最前面的 7×7 卷积和 stride=2 的最大池化，早期特征图会缩小得比较快。因此，我们针对 CIFAR-10 采用更合适的配置：
- 第一层卷积改成 3×3，步长为 1。
- 去掉最前面的最大池化层。
- 保留后面四组残差层。

In [9]:

import torch
import torch.nn as nn


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        # 主分支：第一次卷积
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        # 主分支：第二次卷积
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        # 默认不改变输入
        self.shortcut = nn.Identity()

        # 尺寸不一致时，调整捷径分支
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)

        return out

class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.in_channels = 64 #去除了第一层卷积与池化
        self.conv1 = nn.Conv2d(3,64,kernel_size=3,stride=1,padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()

        self.layer1 = self._make_layer(64,2,stride=1)
        self.layer2 = self._make_layer(128,2,stride=2)
        self.layer3 = self._make_layer(256,2,stride=2)
        self.layer4 = self._make_layer(512,2,stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))#全局平均池化，直接变为1*1

        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self,out_channels,blocks,stride):
        layers = []
#可能需要下采样
        layers.append(
            BasicBlock(
                self.in_channels,
                out_channels,
                stride=stride
            )
        )
        self.in_channels = out_channels
        for _ in range(1,blocks):
            layers.append(
                BasicBlock(
                    self.in_channels,
                    out_channels,
                    stride=1
                )
            )

        return nn.Sequential(*layers)

    def forward(self,x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)

        x = torch.flatten(x, 1)

        # [B, 512] → [B, 10]
        x = self.fc(x)

        return x

if __name__ == "__main__":
    model = ResNet18(num_classes=10)

    x = torch.randn(4, 3, 32, 32)
    y = model(x)

    print("输入形状:", x.shape)
    print("输出形状:", y.shape)
    print("参数量:", sum(p.numel() for p in model.parameters()))


输入形状: torch.Size([4, 3, 32, 32])
输出形状: torch.Size([4, 10])
参数量: 11174026
